# MVCBM — diagnosis apendisitis pediatrik (training di Kaggle)

Notebook ini hanya memanggil fungsi dari file `.py` di repo `pediatric-kumadai`. Kode diedit secara lokal, di-push ke GitHub, lalu dijalankan di sini.

Sebelum *Run All*:
1. **Settings → Accelerator → GPU** (T4).
2. **Settings → Internet → On** (untuk `git clone`).
3. **Add Input** → dataset Kaggle (Private) berisi folder `preprocessed_400/` dan `dicts_diagnosis/imputed/final/` (`app_data_train`, `app_data_test`) hasil `prepare_data.py`.
4. Jika repo GitHub private: **Add-ons → Secrets** → tambahkan `GITHUB_TOKEN` (personal access token dengan akses baca repo).

Output (log, checkpoint, kurva ROC/PR, ringkasan metrik) tersimpan di `/kaggle/working/runs/`.

## 1. Ambil kode dari GitHub

In [4]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/akhyaruu/pediatric-kumadai.git"
BRANCH = "main"
REPO_DIR = "/tmp/pediatric-kumadai"  # outside /kaggle/working, so the code is not saved as notebook output

token = None
try:
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass  # public repository or no secret attached
auth_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", auth_url, BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, auth_url, REPO_DIR], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL], check=True)  # no token on disk
subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=commit %h  %ci  %s"], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

Cloning into '/tmp/pediatric-kumadai'...


commit 4360d16  2026-11-19 21:27:58 +0900  feat: change to five concept


## 2. Paket yang belum ada di image Kaggle

In [5]:
import importlib.util

missing = [pip_name for module, pip_name in [("progressbar", "progressbar2")] if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
print("missing packages installed:", missing or "none")

missing packages installed: none


## 3. Cek GPU, data, dan bobot ResNet-18

In [6]:
import kaggle_setup

kaggle_setup.print_environment()
DATA = kaggle_setup.find_data()
for key, path in DATA.items():
    print(f"{key:15s} {path}")
kaggle_setup.ensure_resnet_weights()

python 3.13.16 (/usr/bin/python3)
torch 2.11.0+cu130, CUDA build 13.0
GPU: Tesla T4, 14.6 GiB, compute capability 7.5
images          /kaggle/input/datasets/akhyaruu/pediatric/preprocessed_400
dict_file       /kaggle/input/datasets/akhyaruu/pediatric/dicts_diagnosis/imputed/final/app_data_train
dict_file_test  /kaggle/input/datasets/akhyaruu/pediatric/dicts_diagnosis/imputed/final/app_data_test
test_images     /kaggle/input/datasets/akhyaruu/pediatric/preprocessed_400
ResNet-18 weights: /tmp/pediatric-kumadai/pretrained_models/resnet/resnet18-5c106cde.pth


'/tmp/pediatric-kumadai/pretrained_models/resnet/resnet18-5c106cde.pth'

## 4. Training + testing MVCBM

`CONFIGS`: setiap config dijalankan untuk setiap seed di `SEEDS`, output ke `/kaggle/working/runs/<nama config>/seed<N>`. Kedua config memakai dictionary yang sama (5 konsep); config 2 konsep hanya memakai 2 konsep pertama, jadi perbandingannya adil. Paper melaporkan rata-rata beberapa seed. `OVERRIDES` dapat dipakai untuk mengubah isi config tanpa mengedit file, mis. `{"c_epochs": 2, "t_epochs": 2}` untuk uji cepat.

In [7]:
import importlib

import train_mvcbm

importlib.reload(train_mvcbm)  # picks up a new version of the code after re-running step 1
importlib.reload(kaggle_setup)

CONFIGS = [
    "configs/mvcbm_seq_lstm_diag.yaml",  # 2 concepts
    "configs/mvcbm_seq_lstm_diag_5c.yaml",  # 5 concepts
]
SEEDS = [42]
OVERRIDES = {}

results = {}  # results[run][seed] = test metrics of the last epoch
for config_path in CONFIGS:
    run = os.path.splitext(os.path.basename(config_path))[0]
    results.setdefault(run, {})
    for seed in SEEDS:
        overrides = {**kaggle_setup.kaggle_overrides(DATA, run_name=f"{run}/seed{seed}"), **OVERRIDES}
        config, results[run][seed] = train_mvcbm.run(config_path, seed=seed, overrides=overrides)
        print(f"\n{run}, seed {seed}: outputs in {config['log_directory']}\n")

Length of training array 520
Length of test array 58
Train target class distribution:  Counter({np.float64(1.0): 344, np.float64(0.0): 176})
Train concepts class distribution: 
... Counter({1.0: 399, 0.0: 121})
... Counter({0.0: 293, 1.0: 227})
Test target class distribution:  Counter({np.float64(1.0): 38, np.float64(0.0): 20})

STARTING FINAL MODEL TRAINING!


Starting concepts training!



Epoch 20/20: 100%|██████████| 520/520 [00:29<00:00, 17.37img/s, Concepts loss=1.01, Target loss=0.688, Total loss=1.7, lr=0.0001]  


(Mode = c) Epoch validation scores:
        Concepts summed loss: 1.2784722529608628
            Concept 0 loss: 0.575578844752805
            Concept 1 loss: 0.7028934071803915
        Concepts averaged metrics
            Accuracy: 0.6293103448275862 F1-macro: 0.6189599616070205
            AUROC: 0.6945261437908496 AUPR: 0.6880717960310372
        Target
            Target loss: 0.6801300747641201
            PPV: 0.6551724137931034 NPV: 0.0 
            Sensitivity: 1.0 Specificity: 0.0
            Accuracy: 0.6551724137931034 Balanced accuracy: 0.5
            F1_1: 0.7916666666666666 F1_0: 0.0 F1_macro: 0.3958333333333333
            AUROC: 0.6039473684210527 AUPR: 0.6673986674402056
        Total loss: 1.9586023544443065

Starting target training!



Epoch 20/20: 100%|██████████| 520/520 [00:29<00:00, 17.92img/s, Concepts loss=0.925, Target loss=0.634, Total loss=1.56, lr=0.01]


(Mode = t) Epoch validation scores:
        Concepts summed loss: 1.2784722529608628
            Concept 0 loss: 0.575578844752805
            Concept 1 loss: 0.7028934071803915
        Concepts averaged metrics
            Accuracy: 0.6293103448275862 F1-macro: 0.6189599616070205
            AUROC: 0.6945261437908496 AUPR: 0.6880717960310372
        Target
            Target loss: 0.6806486341459997
            PPV: 0.78125 NPV: 0.5 
            Sensitivity: 0.6578947368421053 Specificity: 0.65
            Accuracy: 0.6551724137931034 Balanced accuracy: 0.6539473684210526
            F1_1: 0.7142857142857143 F1_0: 0.5652173913043478 F1_macro: 0.639751552795031
            AUROC: 0.6039473684210526 AUPR: 0.6684129990515675
        Total loss: 1.9591209066325221

Training finished, model saved!

EVALUATION ON THE TEST SET:

Summed concepts loss on last epoch: 0.7028934071803915
(Concept 0) Test results on last epoch: 
    sc0_loss: 0.575578844752805
    sc0_accuracy: 0.7931034482758621


Epoch 20/20: 100%|██████████| 520/520 [00:29<00:00, 17.77img/s, Concepts loss=2.6, Target loss=0.694, Total loss=3.3, lr=0.0001]  


(Mode = c) Epoch validation scores:
        Concepts summed loss: 2.6528594658292572
            Concept 0 loss: 0.4394261998863056
            Concept 1 loss: 0.7265478906960323
            Concept 2 loss: 0.45371245618524225
            Concept 3 loss: 0.505306796780948
            Concept 4 loss: 0.5278661086641508
        Concepts averaged metrics
            Accuracy: 0.7241379310344828 F1-macro: 0.6057290965464979
            AUROC: 0.6585077749048338 AUPR: 0.48685542282853406
        Target
            Target loss: 0.7064828173867588
            PPV: 0.42857142857142855 NPV: 0.3137254901960784 
            Sensitivity: 0.07894736842105263 Specificity: 0.8
            Accuracy: 0.3275862068965517 Balanced accuracy: 0.43947368421052635
            F1_1: 0.13333333333333333 F1_0: 0.4507042253521127 F1_macro: 0.292018779342723
            AUROC: 0.4171052631578947 AUPR: 0.6069083788183675
        Total loss: 3.359342328433333

Starting target training!



Epoch 20/20: 100%|██████████| 520/520 [00:28<00:00, 18.46img/s, Concepts loss=2.62, Target loss=0.521, Total loss=3.14, lr=0.01]


(Mode = t) Epoch validation scores:
        Concepts summed loss: 2.6528594658292572
            Concept 0 loss: 0.4394261998863056
            Concept 1 loss: 0.7265478906960323
            Concept 2 loss: 0.45371245618524225
            Concept 3 loss: 0.505306796780948
            Concept 4 loss: 0.5278661086641508
        Concepts averaged metrics
            Accuracy: 0.7241379310344828 F1-macro: 0.6057290965464979
            AUROC: 0.6585077749048338 AUPR: 0.48685542282853406
        Target
            Target loss: 0.7885268844407181
            PPV: 0.6842105263157895 NPV: 0.358974358974359 
            Sensitivity: 0.34210526315789475 Specificity: 0.7
            Accuracy: 0.46551724137931033 Balanced accuracy: 0.5210526315789473
            F1_1: 0.45614035087719296 F1_0: 0.4745762711864407 F1_macro: 0.4653583110318168
            AUROC: 0.6697368421052632 AUPR: 0.7536328444657033
        Total loss: 3.44138633793798

Training finished, model saved!

EVALUATION ON THE TEST SE

## 5. Ringkasan metrik test (epoch terakhir)

Konsep (urutan `CONCEPTS` di `prepare_data.py`): `c0` = Appendix_on_US, `c1` = Free_Fluids, `c2` = Appendix_Wall_Layers, `c3` = Target_Sign, `c4` = Surrounding_Tissue_Reaction. Tabel pertama membandingkan config (rata-rata dan, untuk lebih dari satu seed, standar deviasi); di bawahnya detail per seed.

In [8]:
from IPython.display import display

comparison = kaggle_setup.compare_runs(results)
comparison.to_csv("/kaggle/working/runs/comparison.csv")
display(comparison)

for run, results_per_seed in results.items():
    summary = kaggle_setup.summarize(results_per_seed)
    summary.to_csv(f"/kaggle/working/runs/{run}/summary.csv")
    print(run)
    display(summary)

,t_auroc,t_aupr,t_balanced_accuracy,t_f1_macro,c0_auroc,c1_auroc,c2_auroc,c3_auroc,c4_auroc
run,,,,,,,,,
mvcbm_seq_lstm_diag,0.604,0.668,0.654,0.640,0.869,0.520,-,-,-
mvcbm_seq_lstm_diag_5c,0.670,0.754,0.521,0.465,0.907,0.386,0.679,0.590,0.731


mvcbm_seq_lstm_diag


,t_auroc,t_aupr,t_balanced_accuracy,t_f1_macro,c0_auroc,c1_auroc
seed,,,,,,
42,0.603947,0.668413,0.653947,0.639752,0.869444,0.519608


mvcbm_seq_lstm_diag_5c


,t_auroc,t_aupr,t_balanced_accuracy,t_f1_macro,c0_auroc,c1_auroc,c2_auroc,c3_auroc,c4_auroc
seed,,,,,,,,,
42,0.669737,0.753633,0.521053,0.465358,0.906944,0.386029,0.679167,0.589744,0.730655
